# Province classification study - 4 approaches, same data, same test set

Which architecture / training strategy reads the Khmer province line best?
Every run uses the identical split (2,515 / 529 / 567), the identical
augmentation, the identical test-time preprocessing, 40 epochs, seed 42.

| run | architecture | initial weights | what trains | rubric dimension |
|---|---|---|---|---|
| **A** | ResNet18 | random | everything | training strategy |
| **B** | ResNet18 | ImageNet | **only the final layer** (feature extraction / linear probe) | training strategy |
| **C** | ResNet18 | ImageNet | everything, low LR (fine-tuning) | training strategy |
| **D** | small 4-block CNN (0.4 M params) | random | everything | architecture |

Each run writes `results/province_study/<run>/` with `history.csv` (per-epoch
train/val loss + accuracy), `run.json` (params, time, GPU, test accuracy,
macro-F1) and `test_predictions.csv` (every test image with true / predicted
class). The last cell copies that folder to Drive - commit it to the repo.

Run time: roughly 10 min per ResNet18 run on a T4, less for D.


In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'

import os, zipfile, shutil
shutil.rmtree('/content/alpr', ignore_errors=True)
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

In [ ]:
# ---- BUNDLE FRESHNESS CHECK -- do not skip -------------------------------
import glob

tr = open('scripts/recognition/train_province_classifier.py', encoding='utf-8').read()
needed = ['--arch', '--weight-decay', '--seed', '--resume', 'history.csv', 'run.json']
missing = [f for f in needed if f not in tr]
n_train = len(glob.glob('data/province_crops/train/*/*.jpg'))
n_test  = len(glob.glob('data/province_crops/test/*/*.jpg'))

print('missing flags   :', missing or 'none')
print('province crops  : train', n_train, '| test', n_test)
assert not missing, 'STALE BUNDLE -- run make_colab_bundle.py again and re-upload.'
assert n_train >= 2500 and n_test >= 560, 'province crops missing'
print()
print('Bundle is current. Safe to continue.')

In [ ]:
!pip -q install torch torchvision pyyaml tqdm pillow opencv-python-headless numpy

In [ ]:
# ---- RUN A: ResNet18 from scratch ----------------------------------------
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch resnet18 --lr 1e-3 \
    --run-name A_resnet18_scratch --out models/recognition/prov_A_scratch.pth

In [ ]:
# ---- RUN B: ResNet18 feature extraction (ImageNet frozen, head only) -----
# Watch the [freeze] line: ~11.18 M frozen, 13,338 trainable (0.12%).
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch resnet18 --pretrained --freeze --lr 1e-3 \
    --run-name B_resnet18_frozen --out models/recognition/prov_B_featext.pth

In [ ]:
# ---- RUN C: ResNet18 fine-tuning (ImageNet, all layers, low LR) ----------
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch resnet18 --pretrained --lr 1e-4 \
    --run-name C_resnet18_finetune --out models/recognition/prov_C_finetune.pth

In [ ]:
# ---- RUN D: small CNN from scratch (low-capacity baseline) ---------------
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch small_cnn --lr 1e-3 \
    --run-name D_smallcnn_scratch --out models/recognition/prov_D_smallcnn.pth

In [ ]:
# ---- RESULTS TABLE (from run.json) ---------------------------------------
import json, glob
rows = []
for p in sorted(glob.glob('results/province_study/*/run.json')):
    r = json.load(open(p))
    rows.append(r)
print('%-22s %-10s %-9s %12s %8s %9s %9s %8s' % (
    'run', 'arch', 'mode', 'trainable', 'best_ep', 'val_acc', 'test_acc', 'macroF1'))
print('-' * 96)
for r in rows:
    print('%-22s %-10s %-9s %12s %8d %8.2f%% %8.2f%% %7.2f%%  (%.1f min)' % (
        r['run_name'], r['arch'], r['mode'][:9], f"{r['trainable_params']:,}",
        r['best_epoch'], 100*r['best_val_acc'], 100*r['test_acc'],
        100*r['test_macro_f1'], r['train_wall_sec']/60))
print()
print('Deployed model for reference: 96.1% upright on the same 567 crops (test_province_rotation.py).')

### How to read it

* **A vs C** - does ImageNet initialisation help when the whole network trains? If they tie, 2.5 k images are enough to learn from scratch and pretraining only speeds convergence (check `best_epoch`).
* **B** - a frozen ImageNet backbone is a fixed feature extractor. If it collapses, ImageNet features do not separate Khmer glyphs: the low-level filters must be re-learned (domain shift / inductive bias).
* **D vs A** - capacity: does a 0.4 M-param CNN keep up with an 11 M-param ResNet at 128 px?
* Open `history.csv` for each run to see over-fitting (train >> val) or under-fitting (both flat).


In [ ]:
# ---- SAVE EVERYTHING TO DRIVE ---------------------------------------------
# results/province_study/  -> commit this folder to the repo (CSV/JSON only)
# models/recognition/prov_*  -> weights (35-45 MB each) for Drive links
import shutil, os, glob
dst = '/content/drive/MyDrive/ALPR/province_study'
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree('results/province_study', dst,
                ignore=shutil.ignore_patterns('last.pth'))
os.makedirs('/content/drive/MyDrive/ALPR/trained', exist_ok=True)
for f in glob.glob('models/recognition/prov_*'):
    shutil.copy(f, '/content/drive/MyDrive/ALPR/trained/')
print('saved ->', dst)
print('weights ->', sorted(os.listdir('/content/drive/MyDrive/ALPR/trained')))